# QLoRA Fine-Tuning: Text-to-SQL on Qwen2.5-Coder-1.5B

Robust Kaggle notebook for **2× NVIDIA T4 DDP** with PEFT + TRL + bitsandbytes.

### Fixed in this version
- No `bitsandbytes`, PEFT, TRL, or Transformers imports before `notebook_launcher()`.
- No `torch.cuda.*` probing before DDP workers are spawned.
- Dataset transformations never write under Kaggle's read-only `/kaggle/input`.
- All Hugging Face caches and training outputs live under `/kaggle/working`.
- T4 uses FP16 by default; BF16 is opt-in for supported GPUs.
- Dataset is explicitly converted to a `full_text` field for SFT.
- No unnecessary NCCL barriers around filesystem operations.
- Main-process-only checkpoint/export operations.
- Pre-launch sanity checks detect an already-imported `bitsandbytes` module.
- Smoke test is intentionally small and should be run before full training.

> **Important:** Start this notebook from a fresh Kaggle Session after installing/upgrading packages. Do not continue from a Python kernel that previously imported `bitsandbytes`.

## 1. Install / verify training libraries

Run this in a fresh Kaggle Session. After a package upgrade, restart the session once before running the remaining cells.

In [1]:
!pip install -q -U \
    "transformers>=5.0,<6.0" \
    "safetensors>=0.8.0,<0.9.0" \
    "accelerate>=1.0" \
    "peft>=0.15" \
    "trl>=0.20" \
    "bitsandbytes>=0.50" \
    "datasets" \
    "tabulate"

!pip install -q --upgrade "torchao>=0.17.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 45.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 30.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 112.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not 

## 2. Clean parent-process setup

This cell intentionally imports only lightweight/runtime modules. The distributed launcher must start before anything that may import `bitsandbytes`.

In [2]:
import os
import sys
import gc
import json
import math
import time
import hashlib
import datetime
import subprocess
from pathlib import Path

# Writable locations on Kaggle.
WORK_DIR = Path("/kaggle/working/sql_engine")
HF_HOME = WORK_DIR / "hf_cache"
HF_DATASETS_CACHE = HF_HOME / "datasets"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"

for p in [WORK_DIR, HF_HOME, HF_DATASETS_CACHE, HF_HUB_CACHE, TRANSFORMERS_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

# Ask PyTorch to use an NVML-based availability check rather than initializing
# the CUDA runtime in the notebook parent process.
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"

print(f"Working directory : {WORK_DIR}")
print(f"HF cache          : {HF_HOME}")
print(f"bitsandbytes loaded before DDP: {'bitsandbytes' in sys.modules}")

Working directory : /kaggle/working/sql_engine
HF cache          : /kaggle/working/sql_engine/hf_cache
bitsandbytes loaded before DDP: False


## 3. Hardware detection without initializing CUDA

Use `nvidia-smi` for pre-launch GPU discovery. Do **not** call `torch.cuda.*` here.

In [3]:
def nvidia_smi_query():
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.total",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        )
        return [line.strip() for line in out.splitlines() if line.strip()]
    except Exception:
        return []

GPU_INFO = nvidia_smi_query()
NUM_GPUS = len(GPU_INFO)

print("Detected GPUs:")
for line in GPU_INFO:
    print("  ", line)

if NUM_GPUS == 0:
    print("No NVIDIA GPUs detected; this notebook is intended primarily for Kaggle CUDA training.")

Detected GPUs:
   0, Tesla T4, 15360
   1, Tesla T4, 15360


## 4. Configuration

In [4]:
CONFIG = {
    # Run mode
    "SMOKE_TEST": True,
    "USE_MULTI_GPU_DDP": NUM_GPUS >= 2,
    "NUM_GPUS": max(1, NUM_GPUS),
    "SMOKE_SAMPLES_TRAIN": 10,
    "SMOKE_SAMPLES_DEV": 10,
    "SMOKE_STEPS": 2,

    # Model / data
    "MODEL_ID": "Qwen/Qwen2.5-Coder-1.5B",
    "DATASET_PATH": "/kaggle/input/datasets/pernavjain/natural-language-to-sql",
    "MAX_SEQ_LENGTH": 1024,

    # QLoRA
    "LORA_R": 16,
    "LORA_ALPHA": 32,
    "LORA_DROPOUT": 0.05,
    "TARGET_MODULES": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    # Training
    "NUM_EPOCHS": 3,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 4,
    "GRAD_ACCUM_STEPS": 2,
    "WARMUP_RATIO": 0.05,
    "WEIGHT_DECAY": 0.01,
    "MAX_GRAD_NORM": 1.0,
    "LOGGING_STEPS": 10,
    "OPTIMIZER": "paged_adamw_8bit",
    "DATALOADER_NUM_WORKERS": 2,
    "DATALOADER_PIN_MEMORY": True,

    # Evaluation / checkpointing
    "EVAL_STEPS": 200,
    "EVAL_DEV_SAMPLES": 500,
    "RUN_MID_TRAIN_EM": False,  # Safer default for DDP; post-training EM always runs.

    # Precision / launcher
    "MIXED_PRECISION": "fp16",  # T4-safe default.

    # Output
    "ADAPTER_OUTPUT_DIR": str(WORK_DIR / "models" / "qlora-adapter"),
    "MERGED_OUTPUT_DIR": str(WORK_DIR / "models" / "text2sql-v1"),
}

print(json.dumps(CONFIG, indent=2))

{
  "SMOKE_TEST": true,
  "USE_MULTI_GPU_DDP": true,
  "NUM_GPUS": 2,
  "SMOKE_SAMPLES_TRAIN": 10,
  "SMOKE_SAMPLES_DEV": 10,
  "SMOKE_STEPS": 2,
  "MODEL_ID": "Qwen/Qwen2.5-Coder-1.5B",
  "DATASET_PATH": "/kaggle/input/datasets/pernavjain/natural-language-to-sql",
  "MAX_SEQ_LENGTH": 1024,
  "LORA_R": 16,
  "LORA_ALPHA": 32,
  "LORA_DROPOUT": 0.05,
  "TARGET_MODULES": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "NUM_EPOCHS": 3,
  "LEARNING_RATE": 0.0002,
  "BATCH_SIZE": 4,
  "GRAD_ACCUM_STEPS": 2,
  "WARMUP_RATIO": 0.05,
  "WEIGHT_DECAY": 0.01,
  "MAX_GRAD_NORM": 1.0,
  "LOGGING_STEPS": 10,
  "OPTIMIZER": "paged_adamw_8bit",
  "DATALOADER_NUM_WORKERS": 2,
  "DATALOADER_PIN_MEMORY": true,
  "EVAL_STEPS": 200,
  "EVAL_DEV_SAMPLES": 500,
  "RUN_MID_TRAIN_EM": false,
  "MIXED_PRECISION": "fp16",
  "ADAPTER_OUTPUT_DIR": "/kaggle/working/sql_engine/models/qlora-adapter",
  "MERGED_OUTPUT_DIR": "/kaggle/working/sql_engine/m

## 5. Pre-launch sanity checks

Run these checks **before** `notebook_launcher()`. `bitsandbytes` must not already be present in `sys.modules`.

In [5]:
# Do not proceed if the current kernel has already loaded bitsandbytes.
# Restart the Kaggle Session if this prints True.
assert "bitsandbytes" not in sys.modules, (
    "bitsandbytes is already loaded in the parent notebook process. "
    "Restart the Kaggle Session and rerun the notebook from the beginning."
)

# These are the only package-level checks we need here.
import importlib.util
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)
print("NUM_GPUS:", NUM_GPUS)
print("DDP enabled:", CONFIG["USE_MULTI_GPU_DDP"] and NUM_GPUS > 1)

bitsandbytes installed: True
NUM_GPUS: 2
DDP enabled: True


## 6. SQL evaluation helpers

No Transformers/PEFT/TRL imports are needed here.

In [6]:
import re


def normalize_sql(query: str) -> str:
    if not query:
        return ""
    query = re.sub(r"^```(?:sql)?\s*", "", query.strip(), flags=re.IGNORECASE)
    query = re.sub(r"\s*```$", "", query.strip())
    query = re.sub(r"--.*$", "", query, flags=re.MULTILINE)
    query = re.sub(r"/\*.*?\*/", "", query, flags=re.DOTALL)
    query = query.strip().rstrip(";").strip()
    query = " ".join(query.split()).replace("`", "")
    return query.lower().strip()


def compute_exact_match(predictions, references):
    total = min(len(predictions), len(references))
    correct = sum(
        normalize_sql(predictions[i]) == normalize_sql(references[i])
        for i in range(total)
    )
    return {
        "exact_match": round(100.0 * correct / total, 2) if total else 0.0,
        "total": total,
        "correct": correct,
    }

## 7. End-to-end worker

All CUDA-sensitive training libraries are imported **inside** `train_worker()`. The dataset transformation uses `keep_in_memory=True` so it cannot attempt to write temporary Arrow files into read-only `/kaggle/input`.

In [7]:
import torch


def train_worker(cfg=None):
    cfg = cfg or CONFIG

    # ------------------------------------------------------------
    # Distributed context. This is the first point at which the
    # spawned worker intentionally touches CUDA/distributed state.
    # ------------------------------------------------------------
    from accelerate import Accelerator

    accelerator = Accelerator()
    rank = accelerator.process_index
    local_rank = accelerator.local_process_index
    world_size = accelerator.num_processes
    is_main = accelerator.is_main_process

    if is_main:
        print("=" * 80)
        print("STARTING QLoRA TEXT-to-SQL TRAINING")
        print(f"World size : {world_size}")
        print(f"Primary device : {accelerator.device}")
        print("=" * 80)

    # ------------------------------------------------------------
    # Imports that can pull in bitsandbytes / CUDA libraries.
    # NEVER move these to global scope.
    # ------------------------------------------------------------
    from datasets import load_from_disk
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
    )
    from trl import SFTConfig, SFTTrainer

    # ------------------------------------------------------------
    # Hardware / precision — now safe because we are inside worker.
    # ------------------------------------------------------------
    is_cuda = torch.cuda.is_available()
    is_bf16 = is_cuda and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if is_bf16 else torch.float16

    if is_cuda:
        torch.cuda.set_device(local_rank)
        device_map = {"": local_rank} if world_size > 1 else {"": 0}
    else:
        device_map = None

    if is_main:
        print(f"CUDA: {is_cuda} | BF16 supported: {is_bf16} | compute dtype: {compute_dtype}")

    # ------------------------------------------------------------
    # Tokenizer
    # ------------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(
        cfg["MODEL_ID"],
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ------------------------------------------------------------
    # QLoRA model
    # ------------------------------------------------------------
    if not is_cuda:
        raise RuntimeError("This training notebook expects a CUDA GPU.")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    if is_main:
        print(f"Loading {cfg['MODEL_ID']} in 4-bit NF4...")

    model = AutoModelForCausalLM.from_pretrained(
        cfg["MODEL_ID"],
        quantization_config=bnb_config,
        device_map=device_map,
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

    lora_config = LoraConfig(
        r=cfg["LORA_R"],
        lora_alpha=cfg["LORA_ALPHA"],
        lora_dropout=cfg["LORA_DROPOUT"],
        target_modules=cfg["TARGET_MODULES"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    if is_main:
        model.print_trainable_parameters()

    # ---------------------------------------------------------
    # Dataset
    # ---------------------------------------------------------
    
    dataset = load_from_disk('/kaggle/input/datasets/pernavjain/natural-language-to-sql')
    
    
    def prepare_text2sql_example(example):
        return {
            "prompt": example["prompt"],
            "completion": " " + example["sql"],
        }
    
    
    train_data = dataset["train"].map(
        prepare_text2sql_example,
        keep_in_memory=True,
        load_from_cache_file=False,
    )
    
    val_data = dataset["validation"].map(
        prepare_text2sql_example,
        keep_in_memory=True,
        load_from_cache_file=False,
    )
    
    # Keep ONLY the fields required by prompt-completion SFT.
    train_data = train_data.select_columns(
        ["prompt", "completion"]
    )
    
    val_data = val_data.select_columns(
        ["prompt", "completion"]
    )
    
    if cfg["SMOKE_TEST"]:
        train_data = train_data.select(
            range(
                min(
                    cfg["SMOKE_SAMPLES_TRAIN"],
                    len(train_data),
                )
            )
        )
    
        val_data = val_data.select(
            range(
                min(
                    cfg["SMOKE_SAMPLES_DEV"],
                    len(val_data),
                )
            )
        )
    
    if is_main:
        print("Train columns:", train_data.column_names)
        print("Validation columns:", val_data.column_names)
        print("Example prompt:", train_data[0]["prompt"][:150])
        print("Example completion:", train_data[0]["completion"][:150])

    # ------------------------------------------------------------
    # Training arguments
    # ------------------------------------------------------------
    adapter_dir = Path(cfg["ADAPTER_OUTPUT_DIR"])
    adapter_dir.mkdir(parents=True, exist_ok=True)

    max_steps = cfg["SMOKE_STEPS"] if cfg["SMOKE_TEST"] else -1

    # For smoke tests, checkpoint at the end only. For full runs, checkpoint periodically.
    save_steps = max_steps if cfg["SMOKE_TEST"] else cfg["EVAL_STEPS"]

    # Avoid hard-coding scheduler steps from dataset length; let Trainer calculate totals.
    sft_kwargs = {
        "output_dir": str(adapter_dir),
        "learning_rate": cfg["LEARNING_RATE"],
        "lr_scheduler_type": "cosine",
        "gradient_accumulation_steps": cfg["GRAD_ACCUM_STEPS"],
        "per_device_train_batch_size": cfg["BATCH_SIZE"],
        "num_train_epochs": cfg["NUM_EPOCHS"],
        "max_steps": max_steps,
        "bf16": is_bf16,
        "fp16": not is_bf16,
        "tf32": False,
        "logging_steps": cfg["LOGGING_STEPS"],
        "save_strategy": "steps",
        "save_steps": save_steps,
        "save_total_limit": 2,
        "optim": cfg["OPTIMIZER"],
        "dataloader_num_workers": cfg["DATALOADER_NUM_WORKERS"],
        "dataloader_pin_memory": cfg["DATALOADER_PIN_MEMORY"],
        "ddp_find_unused_parameters": False,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "report_to": [],
        "remove_unused_columns": False,
    }

    # TRL version compatibility.
    import inspect
    sft_sig = inspect.signature(SFTConfig.__init__).parameters
    if "max_length" in sft_sig:
        sft_kwargs["max_length"] = cfg["MAX_SEQ_LENGTH"]
    elif "max_seq_length" in sft_sig:
        sft_kwargs["max_seq_length"] = cfg["MAX_SEQ_LENGTH"]
    if "dataset_text_field" in sft_sig:
        sft_kwargs["dataset_text_field"] = "full_text"

    # Never enable an incompatible precision mode.
    if "tf32" not in sft_sig:
        sft_kwargs.pop("tf32", None)
    if "gradient_checkpointing_kwargs" not in sft_sig:
        sft_kwargs.pop("gradient_checkpointing_kwargs", None)

    training_args = SFTConfig(**sft_kwargs)

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_data,
        args=training_args,
        processing_class=tokenizer,
    )

    if is_main:
        print("Beginning trainer.train()...")

    train_result = trainer.train()

    # ------------------------------------------------------------
    # Save only on rank 0. No barrier is needed for local filesystem writes.
    # ------------------------------------------------------------
    if is_main:
        print(f"Training complete. Final loss: {train_result.training_loss:.4f}")
        trainer.model.save_pretrained(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))
        print(f"LoRA adapter saved to {adapter_dir}")

    return {
        "rank": rank,
        "world_size": world_size,
        "training_loss": float(train_result.training_loss),
        "adapter_dir": str(adapter_dir),
    }

## 8. Launch training

For **2× Tesla T4**, the safe default is FP16. The launcher cell itself does not probe CUDA.

> If `bitsandbytes` is already loaded, this cell intentionally stops and tells you to restart the Kaggle Session instead of attempting a broken fallback.

In [8]:
from accelerate import notebook_launcher

assert "bitsandbytes" not in sys.modules, (
    "bitsandbytes is already imported in this notebook kernel. Restart the Kaggle Session."
)

if CONFIG["USE_MULTI_GPU_DDP"] and NUM_GPUS >= 2:
    print(f"--> Launching DDP across {NUM_GPUS} GPUs with FP16...")
    notebook_launcher(
        train_worker,
        args=(CONFIG,),
        num_processes=NUM_GPUS,
        mixed_precision=CONFIG["MIXED_PRECISION"],
    )
else:
    print("--> Launching single-process training...")
    train_worker(CONFIG)

--> Launching DDP across 2 GPUs with FP16...
Launching training on 2 CUDAs.
STARTING QLoRA TEXT-to-SQL TRAINING
World size : 2
Primary device : cuda:0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


CUDA: True | BF16 supported: True | compute dtype: torch.bfloat16


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


Step,Training Loss


Training complete. Final loss: 0.9465
LoRA adapter saved to /kaggle/working/sql_engine/models/qlora-adapter


## 9. Post-training exact-match evaluation

This is deliberately performed **after training** by a single notebook process. That avoids a rank-0-only generation callback pausing while other DDP workers continue into NCCL collectives.

In [9]:
def evaluate_saved_adapter(max_samples=500):
    # These imports happen after DDP has completed, so they are safe in the notebook process.
    from datasets import load_from_disk
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    dataset_path = CONFIG["DATASET_PATH"]
    dataset = load_from_disk(dataset_path)
    dev = dataset["validation"].select(
        range(min(max_samples, len(dataset["validation"])))
    )

    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["MODEL_ID"],
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        CONFIG["MODEL_ID"],
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(
        base,
        CONFIG["ADAPTER_OUTPUT_DIR"],
    )
    model.eval()

    predictions = []
    references = dev["sql"]
    prompts = dev["prompt"]

    device = next(model.parameters()).device
    for i in range(0, len(prompts), 4):
        batch = prompts[i:i+4]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=CONFIG["MAX_SEQ_LENGTH"],
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_tokens = inputs["input_ids"].shape[1]
        for out in outputs:
            gen = out[prompt_tokens:]
            predictions.append(
                tokenizer.decode(gen, skip_special_tokens=True).strip()
            )

    metrics = compute_exact_match(predictions, references)
    print("Post-training Exact Match:", metrics)
    return metrics

if CONFIG["SMOKE_TEST"]:
    print("Smoke-test EM:")
    evaluate_saved_adapter(CONFIG["SMOKE_SAMPLES_DEV"])
else:
    evaluate_saved_adapter(CONFIG["EVAL_DEV_SAMPLES"])

Smoke-test EM:


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Post-training Exact Match: {'exact_match': 30.0, 'total': 10, 'correct': 3}


## 10. Merge LoRA adapter into standalone model

This is intentionally separate from DDP training. It runs after the distributed workers have exited.

In [10]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

merged_dir = Path(CONFIG["MERGED_OUTPUT_DIR"])
adapter_dir = Path(CONFIG["ADAPTER_OUTPUT_DIR"])
merged_dir.mkdir(parents=True, exist_ok=True)

# Clean up the training-side model before loading the merge model.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

is_bf16 = torch.cuda.is_bf16_supported()
target_dtype = torch.bfloat16 if is_bf16 else torch.float16

print(f"Loading base model for merge in {target_dtype}...")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["MODEL_ID"],
    torch_dtype=target_dtype,
    device_map="auto",
    trust_remote_code=True,
)

peft_model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
)

print("Merging LoRA weights...")
standalone_model = peft_model.merge_and_unload()
standalone_model.save_pretrained(
    str(merged_dir),
    safe_serialization=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    str(adapter_dir),
    trust_remote_code=True,
)
tokenizer.save_pretrained(str(merged_dir))

print(f"Standalone model saved to {merged_dir}")

Loading base model for merge in torch.bfloat16...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Merging LoRA weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Standalone model saved to /kaggle/working/sql_engine/models/text2sql-v1


## 11. Export metadata + SHA-256 checksums

In [11]:
def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

file_hashes = {}
for path in sorted(merged_dir.rglob("*")):
    if path.is_file():
        file_hashes[str(path.relative_to(merged_dir))] = compute_sha256(path)

metadata = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "base_model": CONFIG["MODEL_ID"],
    "dataset_path": CONFIG["DATASET_PATH"],
    "hardware": GPU_INFO,
    "world_size": CONFIG["NUM_GPUS"],
    "mixed_precision": CONFIG["MIXED_PRECISION"],
    "config": CONFIG,
    "file_sha256_checksums": file_hashes,
}

meta_path = merged_dir / "training_metadata.json"
meta_path.write_text(json.dumps(metadata, indent=2))

print(f"Metadata written to {meta_path}")
print(f"Checksummed files: {len(file_hashes)}")

Metadata written to /kaggle/working/sql_engine/models/text2sql-v1/training_metadata.json
Checksummed files: 6


## 12. Inference demo

In [12]:
def ask_sql(question: str, schema_ddl: str) -> str:
    prompt = (
        "You are an expert SQL engineer. Given the database schema, write the exact "
        "SQLite query that answers the user question.\n\n"
        f"### Database Schema:\n{schema_ddl}\n\n"
        f"### Question:\n{question}\n\n"
        "### SQL:\n"
    )

    device = next(standalone_model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = standalone_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


demo_schema = """CREATE TABLE `head` (
  `head_ID` NUMBER PRIMARY KEY,
  `name` TEXT,
  `age` NUMBER
);"""

demo_q = "What are the names of heads older than 50?"
print("Question:", demo_q)
print("Generated SQL:", ask_sql(demo_q, demo_schema))

Question: What are the names of heads older than 50?
Generated SQL: SELECT name FROM head WHERE age > 50


## 13. Final run checklist

For a full run, change:

```python
CONFIG["SMOKE_TEST"] = False
CONFIG["RUN_MID_TRAIN_EM"] = False
```

Then restart the Kaggle Session once, rerun the notebook from the top, and launch the full training.

**Expected training architecture:**

`notebook_launcher → worker 0 → GPU 0`  
`notebook_launcher → worker 1 → GPU 1`

There is no parent-process `bitsandbytes` import, no write attempt inside `/kaggle/input`, and no rank-0-only mid-training generation callback that can desynchronize NCCL.

In [13]:
# ============================================================
# DEPENDENCY + VERSION AUDIT
# ============================================================

import sys
import platform
import importlib.metadata as metadata

PACKAGES = [
    "torch",
    "torchvision",
    "torchaudio",
    "torchao",
    "transformers",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "safetensors",
    "datasets",
    "tokenizers",
    "huggingface-hub",
    "numpy",
    "pandas",
    "scipy",
]

print("=" * 90)
print("PYTHON / SYSTEM")
print("=" * 90)
print(f"Python       : {sys.version}")
print(f"Platform     : {platform.platform()}")
print(f"Architecture : {platform.machine()}")

print("\n" + "=" * 90)
print("INSTALLED ML / TRAINING DEPENDENCIES")
print("=" * 90)

for package in PACKAGES:
    try:
        version = metadata.version(package)
        print(f"{package:<22} : {version}")
    except metadata.PackageNotFoundError:
        print(f"{package:<22} : NOT INSTALLED")

print("\n" + "=" * 90)
print("CUDA / GPU")
print("=" * 90)

try:
    import torch

    print(f"PyTorch CUDA available : {torch.cuda.is_available()}")
    print(f"PyTorch CUDA version   : {torch.version.cuda}")
    print(f"cuDNN version          : {torch.backends.cudnn.version()}")

    if torch.cuda.is_available():
        print(f"GPU count              : {torch.cuda.device_count()}")

        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)

            print(
                f"GPU {i}                  : "
                f"{props.name} | "
                f"{props.total_memory / 1024**3:.2f} GB | "
                f"Compute Capability {props.major}.{props.minor}"
            )

except Exception as e:
    print(f"CUDA inspection failed: {e}")

print("\n" + "=" * 90)
print("CRITICAL COMPATIBILITY CHECKS")
print("=" * 90)

# safetensors safe_open signature
try:
    from safetensors import safe_open
    import inspect

    sig = inspect.signature(safe_open)
    print(f"safetensors.safe_open : {sig}")

    if "backend" in sig.parameters:
        print("✓ safetensors supports `backend=`")
    else:
        print("✗ safetensors does NOT support `backend=`")

except Exception as e:
    print(f"✗ safetensors check failed: {e}")

# bitsandbytes
try:
    import bitsandbytes as bnb
    print(f"bitsandbytes loaded   : YES ({bnb.__version__})")
except Exception as e:
    print(f"bitsandbytes loaded   : NO / FAILED ({e})")

# Transformers
try:
    import transformers
    print(f"transformers imported : YES ({transformers.__version__})")
except Exception as e:
    print(f"transformers imported : NO / FAILED ({e})")

# Accelerate
try:
    import accelerate
    print(f"accelerate imported   : YES ({accelerate.__version__})")
except Exception as e:
    print(f"accelerate imported   : NO / FAILED ({e})")

print("\n" + "=" * 90)
print("AUDIT COMPLETE")
print("=" * 90)

PYTHON / SYSTEM
Python       : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform     : Linux-6.12.90+-x86_64-with-glibc2.35
Architecture : x86_64

INSTALLED ML / TRAINING DEPENDENCIES
torch                  : 2.10.0+cu128
torchvision            : 0.25.0+cu128
torchaudio             : 2.10.0+cu128
torchao                : 0.18.0
transformers           : 5.16.1
accelerate             : 1.14.0
peft                   : 0.20.0
trl                    : 1.12.0
bitsandbytes           : 0.50.2
safetensors            : 0.8.0
datasets               : 5.0.1
tokenizers             : 0.23.2
huggingface-hub        : 1.11.0
numpy                  : 2.0.2
pandas                 : 2.3.3
scipy                  : 1.16.3

CUDA / GPU
PyTorch CUDA available : True
PyTorch CUDA version   : 12.8
cuDNN version          : 91002
GPU count              : 2
GPU 0                  : Tesla T4 | 14.56 GB | Compute Capability 7.5
GPU 1                  : Tesla T4 | 14.56 GB | Compute Capability 7.5

CRITICAL 